<a href="https://colab.research.google.com/github/VarshaP-0405/NLP-Skill-Hometask/blob/main/Smart_Next_Word_Predictor_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets

In [3]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

print(dataset)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [4]:
text = "\n".join(dataset["train"]["text"])

print(text[:500])


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs


In [5]:
!pip install -q datasets

import re
from collections import Counter
from datasets import load_dataset

# Load WikiText-2 corpus
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
text = "\n".join(dataset["train"]["text"])

# Clean and tokenize
text = text.lower()
tokens = re.findall(r"\b[a-z]+\b", text)

print("Total words:", len(tokens))

# Create n-gram frequency tables
unigram = Counter(tokens)
bigram = Counter(zip(tokens[:-1], tokens[1:]))
trigram = Counter(zip(tokens[:-2], tokens[1:-1], tokens[2:]))

# Calculate probabilities
total_words = len(tokens)

def predict_next(sentence, top_n=5):
    words = re.findall(r"\b[a-z]+\b", sentence.lower())

    if not words:
        return []

    candidates = {}

    # Use trigram
    if len(words) >= 2:
        previous = (words[-2], words[-1])

        for (w1, w2, w3), count in trigram.items():
            if (w1, w2) == previous:
                candidates[w3] = count / bigram[(w1, w2)]

    # Use bigram if no trigram candidates
    if not candidates:
        previous = words[-1]

        for (w1, w2), count in bigram.items():
            if w1 == previous:
                candidates[w2] = count / unigram[w1]

    # Use unigram if no bigram candidates
    if not candidates:
        for word, count in unigram.most_common(top_n):
            candidates[word] = count / total_words

    # Rank candidates by probability
    predictions = sorted(
        candidates.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return predictions[:top_n]


# User input
sentence = input("Enter a sentence: ")

# Predict next words
predictions = predict_next(sentence, 5)

# Display predictions
print("\nTop predicted next words:")

for i, (word, probability) in enumerate(predictions, 1):
    print(f"{i}. {word} - {probability:.3f}")

Total words: 1679656
Enter a sentence: Machine Learning is

Top predicted next words:
1. a - 0.122
2. the - 0.079
3. not - 0.030
4. also - 0.026
5. an - 0.023
